In [ ]:
# DUNE Model for Temperature Prediction - Rangpur
# Deep UNet++ based Ensemble Approach for Climate Forecasting
# Predicting March-July 2024 temperatures using 1995-2023 data
# Adapted from: DUNE: A Machine Learning Deep UNet++ Based Ensemble Approach

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import os
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Kaggle paths
KAGGLE_WORKING = '/kaggle/working'
IS_KAGGLE = os.path.exists('/kaggle')
OUTPUT_DIR = KAGGLE_WORKING if IS_KAGGLE else '.'

print(f"Running on: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Output directory: {OUTPUT_DIR}")

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
# Load and preprocess data
DATA_PATH = '/kaggle/input/accum-40-yrs-sm-tp-t2m/instant_all.nc'

# Load NetCDF using xarray
ds = xr.open_dataset(DATA_PATH)

print("Dataset variables:", list(ds.data_vars))
print("Dataset dimensions:", dict(ds.dims))

# Extract coordinates
times = pd.to_datetime(ds['valid_time'].values)
lats = ds['latitude'].values
lons = ds['longitude'].values

# Extract variables (t2m and swvl1 as specified)
t2m = ds['t2m'].values
swvl1 = ds['swvl1'].values

# Convert temperature from Kelvin to Celsius
t2m_units = ds['t2m'].attrs.get('units', 'K')
if t2m_units == 'K':
    t2m = t2m - 273.15
    print("Temperature converted from Kelvin to Celsius")

# Rangpur coordinates
target_lat = 25.7439
target_lon = 89.2752

# Find nearest indices
lat_idx = int(np.abs(lats - target_lat).argmin())
lon_idx = int(np.abs(lons - target_lon).argmin())

print(f"Target: lat={target_lat}, lon={target_lon}")
print(f"Nearest: lat={lats[lat_idx]:.2f}, lon={lons[lon_idx]:.2f}")

# Extract time series for Rangpur
temp_series = t2m[:, lat_idx, lon_idx]
sm_series = swvl1[:, lat_idx, lon_idx]

# Handle NaN values
temp_series = np.nan_to_num(temp_series, nan=np.nanmean(temp_series))
sm_series = np.nan_to_num(sm_series, nan=np.nanmean(sm_series))

# Create DataFrame
df = pd.DataFrame({
    'time': times,
    'temperature': temp_series.astype(float),
    'soil_moisture': sm_series.astype(float)
})

df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day

ds.close()

print(f"\nFull dataset shape: {df.shape}")
print(f"Date range: {df['time'].min()} to {df['time'].max()}")


In [ ]:
# Filter for March to July data (months 3-7) from 1995 to 2024
TARGET_MONTHS = [3, 4, 5, 6, 7]  # March to July
MONTH_NAMES = {3: 'March', 4: 'April', 5: 'May', 6: 'June', 7: 'July'}

df_filtered = df[(df['month'].isin(TARGET_MONTHS)) & 
                 (df['year'] >= 1995) & 
                 (df['year'] <= 2024)].copy()
df_filtered = df_filtered.dropna()
df_filtered = df_filtered.sort_values('time').reset_index(drop=True)

print(f"Filtered dataset shape: {df_filtered.shape}")
print(f"Years covered: {df_filtered['year'].min()} to {df_filtered['year'].max()}")
print(f"Months covered: {sorted(df_filtered['month'].unique())}")
print(f"\nData points per year:")
print(df_filtered.groupby('year').size())

# Data split based on available years (1995-2024)
# Training: 1995-2019 (25 years)
# Validation: 2020-2022 (3 years)
# Testing: 2023-2024 (2 years, but we predict 2024 March-July)

TRAIN_YEARS = list(range(1995, 2020))  # 1995-2019
VAL_YEARS = list(range(2020, 2023))     # 2020-2022
TEST_YEARS = [2023, 2024]               # 2023-2024

df_train = df_filtered[df_filtered['year'].isin(TRAIN_YEARS)].copy()
df_val = df_filtered[df_filtered['year'].isin(VAL_YEARS)].copy()
df_test_2024 = df_filtered[df_filtered['year'] == 2024].copy()

# For testing, we also need 2023 data as context
df_test_context = df_filtered[df_filtered['year'] == 2023].copy()

print(f"\n{'='*60}")
print("Data Split Summary")
print(f"{'='*60}")
print(f"Training data ({TRAIN_YEARS[0]}-{TRAIN_YEARS[-1]}): {df_train.shape[0]} samples")
print(f"Validation data ({VAL_YEARS[0]}-{VAL_YEARS[-1]}): {df_val.shape[0]} samples")
print(f"Test context (2023): {df_test_context.shape[0]} samples")
print(f"Test target (2024): {df_test_2024.shape[0]} samples")

# Show monthly distribution for 2024
print(f"\n2024 Test Data by Month:")
for month in TARGET_MONTHS:
    count = len(df_test_2024[df_test_2024['month'] == month])
    print(f"  {MONTH_NAMES[month]}: {count} samples")


In [ ]:
# Calculate climatology (1995-2019 mean for each month-day combination)
# This follows DUNE approach of calculating anomalies from climatology

CLIMATOLOGY_YEARS = list(range(1995, 2020))  # 25 years for robust climatology

df_climatology_base = df_filtered[df_filtered['year'].isin(CLIMATOLOGY_YEARS)].copy()

# Calculate daily climatology for each month-day
climatology = df_climatology_base.groupby(['month', 'day']).agg({
    'temperature': ['mean', 'std'],
    'soil_moisture': ['mean', 'std']
}).reset_index()

climatology.columns = ['month', 'day', 'temp_clim_mean', 'temp_clim_std', 
                       'sm_clim_mean', 'sm_clim_std']

print("Climatology calculated (1995-2019)")
print(f"Climatology entries: {len(climatology)}")
print(climatology.head(10))

# Function to get climatology value
def get_climatology(row, climatology_df, variable='temp'):
    clim_row = climatology_df[(climatology_df['month'] == row['month']) & 
                               (climatology_df['day'] == row['day'])]
    if len(clim_row) > 0:
        if variable == 'temp':
            return clim_row['temp_clim_mean'].values[0]
        else:
            return clim_row['sm_clim_mean'].values[0]
    return np.nan

# Calculate anomalies from climatology (DUNE approach)
def calculate_anomalies(df_input, climatology_df):
    df_out = df_input.copy()
    df_out['temp_clim'] = df_out.apply(lambda x: get_climatology(x, climatology_df, 'temp'), axis=1)
    df_out['sm_clim'] = df_out.apply(lambda x: get_climatology(x, climatology_df, 'sm'), axis=1)
    df_out['temp_anomaly'] = df_out['temperature'] - df_out['temp_clim']
    df_out['sm_anomaly'] = df_out['soil_moisture'] - df_out['sm_clim']
    return df_out

# Apply anomaly calculation
df_train = calculate_anomalies(df_train, climatology)
df_val = calculate_anomalies(df_val, climatology)
df_test_context = calculate_anomalies(df_test_context, climatology)
df_test_2024 = calculate_anomalies(df_test_2024, climatology)

print("\nAnomalies calculated for all datasets")
print(f"Training anomaly stats:\n{df_train[['temp_anomaly', 'sm_anomaly']].describe()}")


In [ ]:
# Prepare features and normalize data
# Features: temperature anomaly, soil moisture anomaly, month (cyclic), day (cyclic)

def add_cyclic_features(df_input):
    """Add cyclic encoding for month and day features"""
    df_out = df_input.copy()
    # Month cyclic encoding (period = 12)
    df_out['month_sin'] = np.sin(2 * np.pi * df_out['month'] / 12)
    df_out['month_cos'] = np.cos(2 * np.pi * df_out['month'] / 12)
    # Day cyclic encoding (period = 31)
    df_out['day_sin'] = np.sin(2 * np.pi * df_out['day'] / 31)
    df_out['day_cos'] = np.cos(2 * np.pi * df_out['day'] / 31)
    return df_out

# Add cyclic features to all datasets
df_train = add_cyclic_features(df_train)
df_val = add_cyclic_features(df_val)
df_test_context = add_cyclic_features(df_test_context)
df_test_2024 = add_cyclic_features(df_test_2024)

# Define features for the model
# Input features: temp_anomaly, sm_anomaly, month_sin, month_cos, day_sin, day_cos
FEATURES = ['temp_anomaly', 'sm_anomaly', 'month_sin', 'month_cos', 'day_sin', 'day_cos']
TARGET = 'temp_anomaly'  # Predict temperature anomaly

# Initialize scalers
scaler_X = MinMaxScaler(feature_range=(-1, 1))  # Scale to [-1, 1] for better gradient flow
scaler_y = MinMaxScaler(feature_range=(-1, 1))

# Fit scalers on training data only
X_train_scaled = scaler_X.fit_transform(df_train[FEATURES].values)
y_train_scaled = scaler_y.fit_transform(df_train[[TARGET]].values)

# Transform validation and test data
X_val_scaled = scaler_X.transform(df_val[FEATURES].values)
y_val_scaled = scaler_y.transform(df_val[[TARGET]].values)

X_test_context_scaled = scaler_X.transform(df_test_context[FEATURES].values)
y_test_context_scaled = scaler_y.transform(df_test_context[[TARGET]].values)

X_test_2024_scaled = scaler_X.transform(df_test_2024[FEATURES].values)
y_test_2024_scaled = scaler_y.transform(df_test_2024[[TARGET]].values)

print(f"Features: {FEATURES}")
print(f"Target: {TARGET}")
print(f"\nScaled data shapes:")
print(f"  Training X: {X_train_scaled.shape}, y: {y_train_scaled.shape}")
print(f"  Validation X: {X_val_scaled.shape}, y: {y_val_scaled.shape}")
print(f"  Test context X: {X_test_context_scaled.shape}")
print(f"  Test 2024 X: {X_test_2024_scaled.shape}")

# Save scalers and climatology for later use
checkpoint_data = {
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
    'climatology': climatology,
    'features': FEATURES,
    'target': TARGET
}
with open(os.path.join(OUTPUT_DIR, 'dune_data_checkpoint.pkl'), 'wb') as f:
    pickle.dump(checkpoint_data, f)
print(f"\nData checkpoint saved")


In [ ]:
# Create sequences for the DUNE model
# DUNE uses temporal context - we create sequences of past observations

SEQ_LENGTH = 30  # Number of previous time steps to use as input

def create_sequences(X, y, seq_length):
    """Create sequences for time series forecasting"""
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

# Create training sequences
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, SEQ_LENGTH)

# Create validation sequences
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val_scaled, SEQ_LENGTH)

print(f"Sequence length: {SEQ_LENGTH}")
print(f"Training sequences: X={X_train_seq.shape}, y={y_train_seq.shape}")
print(f"Validation sequences: X={X_val_seq.shape}, y={y_val_seq.shape}")

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_seq).to(device)
y_train_tensor = torch.FloatTensor(y_train_seq).to(device)
X_val_tensor = torch.FloatTensor(X_val_seq).to(device)
y_val_tensor = torch.FloatTensor(y_val_seq).to(device)

# Create DataLoaders
BATCH_SIZE = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nBatch size: {BATCH_SIZE}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


In [ ]:
# ============================================================================
# DUNE Model Architecture - Adapted for 1D Time Series
# Deep UNet++ based Ensemble Approach
# Original paper: Shukla & Halem (2024) - arXiv:2408.06262
# ============================================================================

class ResidualBlock1D(nn.Module):
    """
    Residual CNN block with skip connections for 1D time series
    Adapted from DUNE's ResidualBlock for spatial data
    """
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        
        padding = kernel_size // 2
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        # Skip connection (1x1 conv if channels differ)
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        identity = self.skip(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out = out + identity  # Residual connection
        out = self.relu(out)
        
        return out


class DUNE1D(nn.Module):
    """
    Deep UNet++ based Ensemble Model for 1D Time Series
    
    Architecture:
    - Encoder path with residual blocks and average pooling
    - Bottleneck layer
    - Decoder path with skip connections (UNet++ style)
    - Multi-scale ensemble outputs from different decoder levels
    
    Input shape: (batch, seq_length, features) -> transpose to (batch, features, seq_length)
    Output shape: (batch, 1)
    """
    def __init__(self, in_channels=6, seq_length=30, base_filters=32):
        super(DUNE1D, self).__init__()
        
        self.in_channels = in_channels
        self.seq_length = seq_length
        
        # Encoder path
        self.enc1 = ResidualBlock1D(in_channels, base_filters)       # 30 -> 30
        self.pool1 = nn.AvgPool1d(2)                                  # 30 -> 15
        
        self.enc2 = ResidualBlock1D(base_filters, base_filters*2)     # 15 -> 15
        self.pool2 = nn.AvgPool1d(2)                                  # 15 -> 7
        
        self.enc3 = ResidualBlock1D(base_filters*2, base_filters*4)   # 7 -> 7
        self.pool3 = nn.AvgPool1d(2)                                  # 7 -> 3
        
        # Bottleneck
        self.bottleneck = ResidualBlock1D(base_filters*4, base_filters*8)  # 3 -> 3
        
        # Decoder path with skip connections (UNet++ style nested connections)
        self.up3 = nn.ConvTranspose1d(base_filters*8, base_filters*4, 2, stride=2)  # 3 -> 6
        self.dec3 = ResidualBlock1D(base_filters*8, base_filters*4)  # concat with enc3
        
        self.up2 = nn.ConvTranspose1d(base_filters*4, base_filters*2, 2, stride=2)  # 6 -> 12
        self.dec2 = ResidualBlock1D(base_filters*4, base_filters*2)  # concat with enc2
        
        self.up1 = nn.ConvTranspose1d(base_filters*2, base_filters, 2, stride=2)    # 12 -> 24
        self.dec1 = ResidualBlock1D(base_filters*2, base_filters)    # concat with enc1
        
        # Ensemble output heads at different scales (DUNE's multi-scale ensemble)
        # Calculate output sizes after decoder
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Output heads from different decoder levels for ensemble
        self.out1 = nn.Linear(base_filters, 1)      # From dec1 (finest)
        self.out2 = nn.Linear(base_filters*2, 1)    # From dec2
        self.out3 = nn.Linear(base_filters*4, 1)    # From dec3
        self.out4 = nn.Linear(base_filters*8, 1)    # From bottleneck (coarsest)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        # Input: (batch, seq_length, features)
        # Transpose to (batch, features, seq_length) for Conv1d
        x = x.transpose(1, 2)
        
        # Encoder
        e1 = self.enc1(x)                           # (batch, 32, 30)
        e2 = self.enc2(self.pool1(e1))              # (batch, 64, 15)
        e3 = self.enc3(self.pool2(e2))              # (batch, 128, 7)
        
        # Bottleneck
        b = self.bottleneck(self.pool3(e3))         # (batch, 256, 3)
        
        # Decoder with skip connections
        # Handle size mismatches with padding
        d3_up = self.up3(b)                         # (batch, 128, 6)
        # Pad or crop to match e3 size
        if d3_up.size(2) != e3.size(2):
            d3_up = F.pad(d3_up, (0, e3.size(2) - d3_up.size(2)))
        d3 = self.dec3(torch.cat([d3_up, e3], dim=1))  # (batch, 128, 7)
        
        d2_up = self.up2(d3)                        # (batch, 64, 14)
        if d2_up.size(2) != e2.size(2):
            d2_up = F.pad(d2_up, (0, e2.size(2) - d2_up.size(2)))
        d2 = self.dec2(torch.cat([d2_up, e2], dim=1))  # (batch, 64, 15)
        
        d1_up = self.up1(d2)                        # (batch, 32, 30)
        if d1_up.size(2) != e1.size(2):
            d1_up = F.pad(d1_up, (0, e1.size(2) - d1_up.size(2)))
        d1 = self.dec1(torch.cat([d1_up, e1], dim=1))  # (batch, 32, 30)
        
        # Global pooling for each decoder level
        g1 = self.global_pool(d1).squeeze(-1)       # (batch, 32)
        g2 = self.global_pool(d2).squeeze(-1)       # (batch, 64)
        g3 = self.global_pool(d3).squeeze(-1)       # (batch, 128)
        g4 = self.global_pool(b).squeeze(-1)        # (batch, 256)
        
        # Apply dropout
        g1 = self.dropout(g1)
        g2 = self.dropout(g2)
        g3 = self.dropout(g3)
        g4 = self.dropout(g4)
        
        # Ensemble outputs from different scales
        out1 = self.out1(g1)  # Finest scale
        out2 = self.out2(g2)
        out3 = self.out3(g3)
        out4 = self.out4(g4)  # Coarsest scale
        
        # Ensemble average (DUNE's core approach)
        ensemble_output = (out1 + out2 + out3 + out4) / 4
        
        return ensemble_output, [out1, out2, out3, out4]


# Initialize the model
INPUT_CHANNELS = len(FEATURES)
model = DUNE1D(in_channels=INPUT_CHANNELS, seq_length=SEQ_LENGTH, base_filters=32).to(device)

print("DUNE1D Model Architecture:")
print("=" * 70)
print(model)
print("=" * 70)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# ============================================================================
# Loss Function - Adapted from DUNE's Latitude-Weighted RMSE
# For single location (Rangpur), we use a standard RMSE with ensemble weighting
# ============================================================================

class EnsembleRMSELoss(nn.Module):
    """
    RMSE Loss with ensemble member weighting
    Adapted from DUNE's LatitudeWeightedRMSE for single-location forecasting
    
    This loss:
    1. Calculates RMSE for the ensemble average
    2. Adds regularization for ensemble member consistency
    """
    def __init__(self, ensemble_weight=0.1):
        super(EnsembleRMSELoss, self).__init__()
        self.ensemble_weight = ensemble_weight
        
    def forward(self, ensemble_output, ensemble_members, target):
        # Main RMSE loss on ensemble average
        mse = (ensemble_output - target) ** 2
        rmse = torch.sqrt(mse.mean() + 1e-8)
        
        # Ensemble consistency regularization
        # Penalize large variance among ensemble members
        if ensemble_members is not None and len(ensemble_members) > 0:
            members_stack = torch.stack(ensemble_members, dim=0)  # (4, batch, 1)
            ensemble_var = members_stack.var(dim=0).mean()
            consistency_loss = self.ensemble_weight * torch.sqrt(ensemble_var + 1e-8)
        else:
            consistency_loss = 0.0
        
        total_loss = rmse + consistency_loss
        
        return total_loss, rmse, consistency_loss


class CombinedLoss(nn.Module):
    """
    Combined loss function: RMSE + MAE for robust training
    """
    def __init__(self, rmse_weight=0.7, mae_weight=0.3, ensemble_weight=0.1):
        super(CombinedLoss, self).__init__()
        self.rmse_weight = rmse_weight
        self.mae_weight = mae_weight
        self.ensemble_weight = ensemble_weight
        
    def forward(self, ensemble_output, ensemble_members, target):
        # RMSE
        mse = (ensemble_output - target) ** 2
        rmse = torch.sqrt(mse.mean() + 1e-8)
        
        # MAE
        mae = torch.abs(ensemble_output - target).mean()
        
        # Ensemble consistency
        if ensemble_members is not None and len(ensemble_members) > 0:
            members_stack = torch.stack(ensemble_members, dim=0)
            ensemble_var = members_stack.var(dim=0).mean()
            consistency_loss = self.ensemble_weight * torch.sqrt(ensemble_var + 1e-8)
        else:
            consistency_loss = 0.0
        
        total_loss = self.rmse_weight * rmse + self.mae_weight * mae + consistency_loss
        
        return total_loss, rmse, mae


# Initialize loss function
criterion = CombinedLoss(rmse_weight=0.7, mae_weight=0.3, ensemble_weight=0.05)
print("Loss function: CombinedLoss (RMSE + MAE + Ensemble Consistency)")


In [ ]:
# ============================================================================
# Training Configuration
# ============================================================================

EPOCHS = 150
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 25
CHECKPOINT_INTERVAL = 10

# Optimizer (Adam as in DUNE)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# Learning rate scheduler (CosineAnnealing as in DUNE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_rmse': [],
    'val_rmse': [],
    'train_mae': [],
    'val_mae': [],
    'lr': []
}

# Early stopping
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None
start_epoch = 0

# Check for existing checkpoint
checkpoint_file = os.path.join(OUTPUT_DIR, 'dune_checkpoint.pth')
if os.path.exists(checkpoint_file):
    print("Loading checkpoint to resume training...")
    checkpoint = torch.load(checkpoint_file, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_loss = checkpoint['best_val_loss']
    best_model_state = checkpoint.get('best_model_state', None)
    print(f"Resumed from epoch {start_epoch}, best val loss: {best_val_loss:.6f}")
else:
    print("No checkpoint found, starting fresh training...")

print(f"\nTraining Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"  Optimizer: Adam")
print(f"  Scheduler: CosineAnnealingLR")


In [ ]:
# ============================================================================
# Training Loop
# ============================================================================

print("\n" + "="*110)
print(f"{'Epoch':^6} | {'Train Loss':^12} | {'Val Loss':^12} | {'Train RMSE':^12} | {'Val RMSE':^12} | {'Train MAE':^12} | {'Val MAE':^12} | {'LR':^10}")
print("="*110)

for epoch in range(start_epoch, EPOCHS):
    # Training phase
    model.train()
    train_losses = []
    train_preds = []
    train_targets = []
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        ensemble_output, ensemble_members = model(X_batch)
        
        # Calculate loss
        loss, rmse, mae = criterion(ensemble_output, ensemble_members, y_batch)
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping (as recommended in DUNE)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_losses.append(loss.item())
        train_preds.extend(ensemble_output.detach().cpu().numpy())
        train_targets.extend(y_batch.detach().cpu().numpy())
    
    # Validation phase
    model.eval()
    val_losses = []
    val_preds = []
    val_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            ensemble_output, ensemble_members = model(X_batch)
            loss, rmse, mae = criterion(ensemble_output, ensemble_members, y_batch)
            
            val_losses.append(loss.item())
            val_preds.extend(ensemble_output.cpu().numpy())
            val_targets.extend(y_batch.cpu().numpy())
    
    # Calculate metrics
    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    
    # Inverse transform for actual temperature metrics
    train_preds_inv = scaler_y.inverse_transform(np.array(train_preds).reshape(-1, 1)).flatten()
    train_targets_inv = scaler_y.inverse_transform(np.array(train_targets).reshape(-1, 1)).flatten()
    val_preds_inv = scaler_y.inverse_transform(np.array(val_preds).reshape(-1, 1)).flatten()
    val_targets_inv = scaler_y.inverse_transform(np.array(val_targets).reshape(-1, 1)).flatten()
    
    train_rmse = np.sqrt(mean_squared_error(train_targets_inv, train_preds_inv))
    val_rmse = np.sqrt(mean_squared_error(val_targets_inv, val_preds_inv))
    train_mae = mean_absolute_error(train_targets_inv, train_preds_inv)
    val_mae = mean_absolute_error(val_targets_inv, val_preds_inv)
    
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_rmse'].append(train_rmse)
    history['val_rmse'].append(val_rmse)
    history['train_mae'].append(train_mae)
    history['val_mae'].append(val_mae)
    history['lr'].append(current_lr)
    
    # Print epoch results
    print(f"{epoch+1:^6} | {train_loss:^12.6f} | {val_loss:^12.6f} | {train_rmse:^12.4f} | {val_rmse:^12.4f} | {train_mae:^12.4f} | {val_mae:^12.4f} | {current_lr:^10.6f}")
    
    # Learning rate scheduler step
    scheduler.step()
    
    # Early stopping check and best model saving
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'best_model_state': best_model_state,
            'history': history
        }, os.path.join(OUTPUT_DIR, 'dune_best_model.pth'))
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break
    
    # Save periodic checkpoint
    if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'best_model_state': best_model_state,
            'history': history
        }, os.path.join(OUTPUT_DIR, 'dune_checkpoint.pth'))
        print(f"  >> Checkpoint saved at epoch {epoch+1}")

print("="*110)
print("Training completed!")

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Loaded best model with validation loss: {best_val_loss:.6f}")


In [ ]:
# ============================================================================
# Plot Training History
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('DUNE Model Training History - Rangpur Temperature Prediction', fontsize=14, fontweight='bold')

# Loss plot
ax1 = axes[0, 0]
ax1.plot(history['train_loss'], label='Training Loss', color='#2E86AB', linewidth=2)
ax1.plot(history['val_loss'], label='Validation Loss', color='#E94F37', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# RMSE plot
ax2 = axes[0, 1]
ax2.plot(history['train_rmse'], label='Training RMSE', color='#2E86AB', linewidth=2)
ax2.plot(history['val_rmse'], label='Validation RMSE', color='#E94F37', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('RMSE (°C)')
ax2.set_title('Training and Validation RMSE')
ax2.legend()
ax2.grid(True, alpha=0.3)

# MAE plot
ax3 = axes[1, 0]
ax3.plot(history['train_mae'], label='Training MAE', color='#2E86AB', linewidth=2)
ax3.plot(history['val_mae'], label='Validation MAE', color='#E94F37', linewidth=2)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('MAE (°C)')
ax3.set_title('Training and Validation MAE')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Learning rate plot
ax4 = axes[1, 1]
ax4.plot(history['lr'], color='#44AF69', linewidth=2)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Learning Rate')
ax4.set_title('Learning Rate Schedule (Cosine Annealing)')
ax4.grid(True, alpha=0.3)
ax4.set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dune_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal Training Metrics:")
print(f"  RMSE: {history['train_rmse'][-1]:.4f}°C")
print(f"  MAE:  {history['train_mae'][-1]:.4f}°C")
print(f"\nFinal Validation Metrics:")
print(f"  RMSE: {history['val_rmse'][-1]:.4f}°C")
print(f"  MAE:  {history['val_mae'][-1]:.4f}°C")


In [ ]:
# ============================================================================
# Generate Predictions for 2024 (March - July)
# ============================================================================

# Combine 2023 data (context) with 2024 data for prediction
# We need context from before each prediction point

# Concatenate context (2023) and test (2024) data
all_test_data = pd.concat([df_test_context, df_test_2024], ignore_index=True)
all_test_data = all_test_data.sort_values('time').reset_index(drop=True)

# Scale the combined data
X_all_test = scaler_X.transform(all_test_data[FEATURES].values)
y_all_test = scaler_y.transform(all_test_data[[TARGET]].values)

# Find the start index of 2024 data
idx_2024_start = len(df_test_context)

print(f"Total test samples (2023 + 2024): {len(all_test_data)}")
print(f"2024 samples start at index: {idx_2024_start}")
print(f"2024 samples: {len(df_test_2024)}")

# Generate predictions for each day of 2024 (March-July)
model.eval()
predictions_2024 = []
actuals_2024 = []
ensemble_predictions = {f'member_{i+1}': [] for i in range(4)}

with torch.no_grad():
    for i in range(len(df_test_2024)):
        # Current prediction index in all_test_data
        pred_idx = idx_2024_start + i
        
        # Get sequence ending at pred_idx (exclusive)
        seq_start = pred_idx - SEQ_LENGTH
        
        if seq_start >= 0:
            # Get input sequence
            X_seq = X_all_test[seq_start:pred_idx]
            X_tensor = torch.FloatTensor(X_seq).unsqueeze(0).to(device)
            
            # Forward pass
            ensemble_output, ensemble_members = model(X_tensor)
            
            # Store predictions
            predictions_2024.append(ensemble_output.cpu().numpy().flatten()[0])
            actuals_2024.append(y_all_test[pred_idx][0])
            
            # Store ensemble member predictions
            for j, member in enumerate(ensemble_members):
                ensemble_predictions[f'member_{j+1}'].append(member.cpu().numpy().flatten()[0])

# Convert to numpy arrays
predictions_2024 = np.array(predictions_2024)
actuals_2024 = np.array(actuals_2024)

# Inverse transform to get actual temperature anomalies
predictions_2024_inv = scaler_y.inverse_transform(predictions_2024.reshape(-1, 1)).flatten()
actuals_2024_inv = scaler_y.inverse_transform(actuals_2024.reshape(-1, 1)).flatten()

# Get corresponding dates and climatology for absolute temperature
pred_dates = df_test_2024['time'].values[:len(predictions_2024)]
pred_temp_clim = df_test_2024['temp_clim'].values[:len(predictions_2024)]

# Convert anomalies back to actual temperatures
pred_temp_actual = predictions_2024_inv + pred_temp_clim
actual_temp_actual = actuals_2024_inv + pred_temp_clim

print(f"\nPredictions generated: {len(predictions_2024)}")
print(f"Date range: {pred_dates[0]} to {pred_dates[-1]}")


In [ ]:
# ============================================================================
# Evaluation Metrics - Following DUNE evaluation approach
# ============================================================================

def calculate_acc(predictions, targets, mean_pred=None, mean_target=None):
    """
    Calculate Anomaly Correlation Coefficient
    """
    if mean_pred is None:
        mean_pred = np.mean(predictions)
    if mean_target is None:
        mean_target = np.mean(targets)
    
    pred_anom = predictions - mean_pred
    target_anom = targets - mean_target
    
    numerator = np.sum(pred_anom * target_anom)
    denominator = np.sqrt(np.sum(pred_anom**2) * np.sum(target_anom**2))
    
    if denominator == 0:
        return 0.0
    
    return numerator / denominator

# Calculate overall metrics for 2024
overall_rmse = np.sqrt(mean_squared_error(actual_temp_actual, pred_temp_actual))
overall_mae = mean_absolute_error(actual_temp_actual, pred_temp_actual)
overall_r2 = r2_score(actual_temp_actual, pred_temp_actual)
overall_mape = np.mean(np.abs((actual_temp_actual - pred_temp_actual) / actual_temp_actual)) * 100
overall_acc = calculate_acc(pred_temp_actual, actual_temp_actual)

# Calculate metrics for anomalies
anomaly_rmse = np.sqrt(mean_squared_error(actuals_2024_inv, predictions_2024_inv))
anomaly_mae = mean_absolute_error(actuals_2024_inv, predictions_2024_inv)
anomaly_acc = calculate_acc(predictions_2024_inv, actuals_2024_inv)

print("="*70)
print("DUNE Model Performance - 2024 (March-July) Predictions")
print("="*70)
print("\nAbsolute Temperature Metrics:")
print(f"  RMSE:  {overall_rmse:.4f}°C")
print(f"  MAE:   {overall_mae:.4f}°C")
print(f"  R²:    {overall_r2:.4f}")
print(f"  MAPE:  {overall_mape:.2f}%")
print(f"  ACC:   {overall_acc:.4f}")
print("\nTemperature Anomaly Metrics:")
print(f"  RMSE:  {anomaly_rmse:.4f}°C")
print(f"  MAE:   {anomaly_mae:.4f}°C")
print(f"  ACC:   {anomaly_acc:.4f}")

# Create results DataFrame
results_df = pd.DataFrame({
    'Date': pd.to_datetime(pred_dates),
    'Actual_Temp': actual_temp_actual,
    'Predicted_Temp': pred_temp_actual,
    'Actual_Anomaly': actuals_2024_inv,
    'Predicted_Anomaly': predictions_2024_inv,
    'Error': pred_temp_actual - actual_temp_actual,
    'Abs_Error': np.abs(pred_temp_actual - actual_temp_actual)
})
results_df['Month'] = results_df['Date'].dt.month
results_df['Month_Name'] = results_df['Month'].map(MONTH_NAMES)

print("\n" + "="*70)
print("Prediction Summary Statistics:")
print("="*70)
print(results_df[['Actual_Temp', 'Predicted_Temp', 'Error', 'Abs_Error']].describe())


In [ ]:
# ============================================================================
# Monthly Performance Analysis
# ============================================================================

# Calculate metrics for each month
monthly_metrics = []

for month in TARGET_MONTHS:
    month_data = results_df[results_df['Month'] == month]
    if len(month_data) > 0:
        m_rmse = np.sqrt(mean_squared_error(month_data['Actual_Temp'], month_data['Predicted_Temp']))
        m_mae = mean_absolute_error(month_data['Actual_Temp'], month_data['Predicted_Temp'])
        m_r2 = r2_score(month_data['Actual_Temp'], month_data['Predicted_Temp']) if len(month_data) > 1 else np.nan
        m_acc = calculate_acc(month_data['Predicted_Temp'].values, month_data['Actual_Temp'].values)
        m_bias = np.mean(month_data['Error'])
        
        monthly_metrics.append({
            'Month': MONTH_NAMES[month],
            'Samples': len(month_data),
            'RMSE (°C)': m_rmse,
            'MAE (°C)': m_mae,
            'R²': m_r2,
            'ACC': m_acc,
            'Bias (°C)': m_bias,
            'Mean_Actual': month_data['Actual_Temp'].mean(),
            'Mean_Pred': month_data['Predicted_Temp'].mean()
        })

monthly_df = pd.DataFrame(monthly_metrics)

print("\n" + "="*90)
print("Monthly Performance Metrics - 2024")
print("="*90)
print(monthly_df.to_string(index=False))

# Create monthly metrics bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('DUNE Model Monthly Performance - 2024 (March-July)', fontsize=14, fontweight='bold')

months = monthly_df['Month'].tolist()
x_pos = np.arange(len(months))

# RMSE by month
ax1 = axes[0, 0]
bars1 = ax1.bar(x_pos, monthly_df['RMSE (°C)'], color='#E94F37', edgecolor='black', alpha=0.8)
ax1.axhline(y=overall_rmse, color='navy', linestyle='--', linewidth=2, label=f'Overall: {overall_rmse:.3f}°C')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(months)
ax1.set_ylabel('RMSE (°C)')
ax1.set_title('RMSE by Month')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, monthly_df['RMSE (°C)']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# MAE by month
ax2 = axes[0, 1]
bars2 = ax2.bar(x_pos, monthly_df['MAE (°C)'], color='#2E86AB', edgecolor='black', alpha=0.8)
ax2.axhline(y=overall_mae, color='navy', linestyle='--', linewidth=2, label=f'Overall: {overall_mae:.3f}°C')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(months)
ax2.set_ylabel('MAE (°C)')
ax2.set_title('MAE by Month')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, monthly_df['MAE (°C)']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# ACC by month
ax3 = axes[1, 0]
bars3 = ax3.bar(x_pos, monthly_df['ACC'], color='#44AF69', edgecolor='black', alpha=0.8)
ax3.axhline(y=overall_acc, color='navy', linestyle='--', linewidth=2, label=f'Overall: {overall_acc:.3f}')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(months)
ax3.set_ylabel('ACC')
ax3.set_title('Anomaly Correlation Coefficient by Month')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars3, monthly_df['ACC']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Bias by month
ax4 = axes[1, 1]
colors = ['#44AF69' if b >= 0 else '#E94F37' for b in monthly_df['Bias (°C)']]
bars4 = ax4.bar(x_pos, monthly_df['Bias (°C)'], color=colors, edgecolor='black', alpha=0.8)
ax4.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(months)
ax4.set_ylabel('Bias (°C)')
ax4.set_title('Prediction Bias by Month')
ax4.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars4, monthly_df['Bias (°C)']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 if val >= 0 else bar.get_height() - 0.05, 
             f'{val:.3f}', ha='center', va='bottom' if val >= 0 else 'top', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dune_monthly_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# Comprehensive Visualization: Time Series for Each Month of 2024
# ============================================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('DUNE Temperature Forecasts - Rangpur, 2024 (March-July)', fontsize=16, fontweight='bold')

# Flatten axes for easier iteration
axes_flat = axes.flatten()

for idx, month in enumerate(TARGET_MONTHS):
    ax = axes_flat[idx]
    month_data = results_df[results_df['Month'] == month].copy()
    
    if len(month_data) > 0:
        days = range(1, len(month_data) + 1)
        
        # Plot actual vs predicted
        ax.plot(days, month_data['Actual_Temp'].values, 'b-', linewidth=2, 
                marker='o', markersize=4, label='Actual', alpha=0.8)
        ax.plot(days, month_data['Predicted_Temp'].values, 'r--', linewidth=2,
                marker='s', markersize=4, label='Predicted', alpha=0.8)
        
        # Fill between
        ax.fill_between(days, month_data['Actual_Temp'].values, 
                        month_data['Predicted_Temp'].values, 
                        alpha=0.2, color='gray')
        
        # Calculate monthly metrics
        m_rmse = np.sqrt(mean_squared_error(month_data['Actual_Temp'], month_data['Predicted_Temp']))
        m_mae = mean_absolute_error(month_data['Actual_Temp'], month_data['Predicted_Temp'])
        
        # Add metrics text
        textstr = f'RMSE: {m_rmse:.2f}°C\nMAE: {m_mae:.2f}°C'
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)
        
        ax.set_xlabel('Day of Month')
        ax.set_ylabel('Temperature (°C)')
        ax.set_title(f'{MONTH_NAMES[month]} 2024', fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
        # Set x-axis ticks
        if len(days) > 15:
            ax.set_xticks(range(1, len(days)+1, 5))

# Remove the extra subplot (6th)
axes_flat[5].axis('off')

# Add overall summary in the 6th position
ax_summary = axes_flat[5]
ax_summary.axis('off')
summary_text = f"""
DUNE Model Summary
{'='*30}

Overall Performance:
  RMSE:  {overall_rmse:.3f}°C
  MAE:   {overall_mae:.3f}°C  
  R²:    {overall_r2:.3f}
  ACC:   {overall_acc:.3f}

Data Range:
  Training: 1995-2019
  Validation: 2020-2022
  Test: 2024 (Mar-Jul)

Model: DUNE1D
  Ensemble Members: 4
  Sequence Length: {SEQ_LENGTH}
  Parameters: {sum(p.numel() for p in model.parameters()):,}
"""
ax_summary.text(0.1, 0.9, summary_text, transform=ax_summary.transAxes, 
                fontsize=11, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dune_monthly_forecasts.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# Statistical Analysis Plots
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('DUNE Model Statistical Analysis - Rangpur, 2024', fontsize=14, fontweight='bold')

# Plot 1: Scatter Plot - Actual vs Predicted
ax1 = axes[0, 0]
scatter = ax1.scatter(actual_temp_actual, pred_temp_actual, 
                      c=results_df['Month'].values, cmap='viridis', 
                      alpha=0.7, edgecolors='black', s=50)
min_val = min(actual_temp_actual.min(), pred_temp_actual.min())
max_val = max(actual_temp_actual.max(), pred_temp_actual.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

# Regression line
z = np.polyfit(actual_temp_actual, pred_temp_actual, 1)
p = np.poly1d(z)
ax1.plot([min_val, max_val], p([min_val, max_val]), 'g-', linewidth=2, 
         label=f'Regression (slope={z[0]:.3f})')

ax1.set_xlabel('Actual Temperature (°C)')
ax1.set_ylabel('Predicted Temperature (°C)')
ax1.set_title(f'Actual vs Predicted (R² = {overall_r2:.4f})', fontsize=12)
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax1)
cbar.set_label('Month')

# Plot 2: Error Distribution
ax2 = axes[0, 1]
errors = results_df['Error'].values
n, bins, patches = ax2.hist(errors, bins=25, color='steelblue', edgecolor='black', 
                            alpha=0.7, density=True)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
ax2.axvline(x=np.mean(errors), color='green', linestyle='-', linewidth=2, 
            label=f'Mean: {np.mean(errors):.2f}°C')

# Fit normal distribution
from scipy import stats
mu, std = np.mean(errors), np.std(errors)
x_norm = np.linspace(errors.min(), errors.max(), 100)
ax2.plot(x_norm, stats.norm.pdf(x_norm, mu, std), 'orange', linewidth=2, 
         label=f'Normal (σ={std:.2f})')

ax2.set_xlabel('Prediction Error (°C)')
ax2.set_ylabel('Density')
ax2.set_title('Error Distribution', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Residual Plot
ax3 = axes[1, 0]
scatter3 = ax3.scatter(pred_temp_actual, errors, c=results_df['Month'].values, 
                       cmap='viridis', alpha=0.7, edgecolors='black', s=50)
ax3.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax3.axhline(y=mu + 2*std, color='orange', linestyle=':', linewidth=1.5, label='±2σ')
ax3.axhline(y=mu - 2*std, color='orange', linestyle=':', linewidth=1.5)
ax3.set_xlabel('Predicted Temperature (°C)')
ax3.set_ylabel('Residual (°C)')
ax3.set_title('Residual Plot', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)
cbar3 = plt.colorbar(scatter3, ax=ax3)
cbar3.set_label('Month')

# Plot 4: Box Plot by Month
ax4 = axes[1, 1]
month_errors = [results_df[results_df['Month'] == m]['Error'].values for m in TARGET_MONTHS]
bp = ax4.boxplot(month_errors, labels=[MONTH_NAMES[m] for m in TARGET_MONTHS], 
                 patch_artist=True)

colors_box = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax4.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax4.set_ylabel('Error (°C)')
ax4.set_title('Error Distribution by Month', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dune_statistical_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# Complete Time Series and Ensemble Analysis
# ============================================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
fig.suptitle('DUNE Temperature Forecasting Analysis - Rangpur, 2024', fontsize=16, fontweight='bold')

# Plot 1: Complete Time Series (March-July 2024)
ax1 = axes[0]
x_range = range(len(results_df))
ax1.plot(x_range, actual_temp_actual, 'b-', linewidth=2, label='Actual', marker='o', markersize=3, alpha=0.8)
ax1.plot(x_range, pred_temp_actual, 'r--', linewidth=2, label='Predicted', marker='s', markersize=3, alpha=0.8)
ax1.fill_between(x_range, actual_temp_actual, pred_temp_actual, alpha=0.2, color='gray')

# Add month separators
cumulative_days = 0
for month in TARGET_MONTHS[:-1]:
    month_count = len(results_df[results_df['Month'] == month])
    cumulative_days += month_count
    ax1.axvline(x=cumulative_days, color='green', linestyle=':', alpha=0.7)
    ax1.text(cumulative_days - month_count//2, ax1.get_ylim()[1], MONTH_NAMES[month], 
             ha='center', fontsize=10, fontweight='bold')

ax1.set_xlabel('Days (March-July 2024)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Complete Temperature Forecast Time Series', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: Prediction Error Over Time
ax2 = axes[1]
colors_error = ['green' if e >= 0 else 'red' for e in results_df['Error'].values]
ax2.bar(x_range, results_df['Error'].values, color=colors_error, alpha=0.7, edgecolor='black', linewidth=0.3)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.axhline(y=np.mean(results_df['Error']), color='blue', linestyle='--', linewidth=2, 
            label=f'Mean Error: {np.mean(results_df["Error"]):.2f}°C')
ax2.axhline(y=np.mean(results_df['Error']) + np.std(results_df['Error']), 
            color='orange', linestyle=':', linewidth=1.5, label=f'±1σ: {np.std(results_df["Error"]):.2f}°C')
ax2.axhline(y=np.mean(results_df['Error']) - np.std(results_df['Error']), 
            color='orange', linestyle=':', linewidth=1.5)

# Add month separators
cumulative_days = 0
for month in TARGET_MONTHS[:-1]:
    month_count = len(results_df[results_df['Month'] == month])
    cumulative_days += month_count
    ax2.axvline(x=cumulative_days, color='purple', linestyle=':', alpha=0.7)

ax2.set_xlabel('Days (March-July 2024)')
ax2.set_ylabel('Prediction Error (°C)')
ax2.set_title('Prediction Error Over Time', fontsize=12, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Plot 3: Cumulative Error and Running RMSE
ax3 = axes[2]
cumulative_abs_error = np.cumsum(np.abs(results_df['Error'].values))
running_rmse = [np.sqrt(np.mean(results_df['Error'].values[:i+1]**2)) for i in range(len(results_df))]

ax3_twin = ax3.twinx()

line1, = ax3.plot(x_range, cumulative_abs_error, 'purple', linewidth=2, label='Cumulative |Error|')
ax3.fill_between(x_range, 0, cumulative_abs_error, alpha=0.2, color='purple')

line2, = ax3_twin.plot(x_range, running_rmse, 'orange', linewidth=2, label='Running RMSE')

ax3.set_xlabel('Days (March-July 2024)')
ax3.set_ylabel('Cumulative Absolute Error (°C)', color='purple')
ax3_twin.set_ylabel('Running RMSE (°C)', color='orange')
ax3.tick_params(axis='y', labelcolor='purple')
ax3_twin.tick_params(axis='y', labelcolor='orange')
ax3.set_title('Cumulative Error and Running RMSE', fontsize=12, fontweight='bold')
ax3.legend(handles=[line1, line2], loc='upper left')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dune_timeseries_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# Ensemble Member Analysis (DUNE's Core Feature)
# ============================================================================

# Inverse transform ensemble member predictions
ensemble_members_inv = {}
for key in ensemble_predictions:
    if len(ensemble_predictions[key]) > 0:
        ensemble_members_inv[key] = scaler_y.inverse_transform(
            np.array(ensemble_predictions[key]).reshape(-1, 1)
        ).flatten() + pred_temp_clim

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('DUNE Ensemble Member Analysis - Rangpur, 2024', fontsize=14, fontweight='bold')

# Plot 1: All ensemble members vs actual
ax1 = axes[0, 0]
x_range = range(len(results_df))
ax1.plot(x_range, actual_temp_actual, 'k-', linewidth=2.5, label='Actual', alpha=0.9)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
for i, (key, values) in enumerate(ensemble_members_inv.items()):
    ax1.plot(x_range, values, '--', linewidth=1.5, alpha=0.6, color=colors[i], 
             label=f'Member {i+1}')
ax1.plot(x_range, pred_temp_actual, 'r-', linewidth=2, label='Ensemble Avg', alpha=0.9)
ax1.set_xlabel('Days (March-July 2024)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Ensemble Members vs Actual Temperature', fontsize=11)
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Ensemble spread (uncertainty)
ax2 = axes[0, 1]
if len(ensemble_members_inv) > 0:
    all_members = np.array(list(ensemble_members_inv.values()))
    ensemble_mean = np.mean(all_members, axis=0)
    ensemble_std = np.std(all_members, axis=0)
    
    ax2.plot(x_range, ensemble_mean, 'b-', linewidth=2, label='Ensemble Mean')
    ax2.fill_between(x_range, ensemble_mean - ensemble_std, ensemble_mean + ensemble_std,
                    alpha=0.3, color='blue', label='±1σ Uncertainty')
    ax2.fill_between(x_range, ensemble_mean - 2*ensemble_std, ensemble_mean + 2*ensemble_std,
                    alpha=0.1, color='blue', label='±2σ Uncertainty')
    ax2.plot(x_range, actual_temp_actual, 'r--', linewidth=1.5, label='Actual', alpha=0.8)

ax2.set_xlabel('Days (March-July 2024)')
ax2.set_ylabel('Temperature (°C)')
ax2.set_title('Ensemble Prediction with Uncertainty Bounds', fontsize=11)
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Member-wise RMSE comparison
ax3 = axes[1, 0]
member_rmses = []
member_names = []
for i, (key, values) in enumerate(ensemble_members_inv.items()):
    rmse = np.sqrt(mean_squared_error(actual_temp_actual, values))
    member_rmses.append(rmse)
    member_names.append(f'Member {i+1}')
member_rmses.append(overall_rmse)
member_names.append('Ensemble Avg')

bars = ax3.bar(member_names, member_rmses, color=colors + ['red'], edgecolor='black', alpha=0.8)
ax3.axhline(y=overall_rmse, color='navy', linestyle='--', linewidth=2, 
            label=f'Ensemble RMSE: {overall_rmse:.3f}°C')
ax3.set_ylabel('RMSE (°C)')
ax3.set_title('RMSE by Ensemble Member', fontsize=11)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, member_rmses):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 4: Ensemble spread over time
ax4 = axes[1, 1]
if len(ensemble_members_inv) > 0:
    ax4.plot(x_range, ensemble_std, 'purple', linewidth=2, label='Ensemble Std Dev')
    ax4.fill_between(x_range, 0, ensemble_std, alpha=0.3, color='purple')
    
    # Add month separators
    cumulative_days = 0
    for month in TARGET_MONTHS[:-1]:
        month_count = len(results_df[results_df['Month'] == month])
        cumulative_days += month_count
        ax4.axvline(x=cumulative_days, color='green', linestyle=':', alpha=0.7)

ax4.set_xlabel('Days (March-July 2024)')
ax4.set_ylabel('Ensemble Std Dev (°C)')
ax4.set_title('Ensemble Uncertainty Over Time', fontsize=11)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dune_ensemble_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print ensemble statistics
print("\n" + "="*60)
print("Ensemble Member Statistics")
print("="*60)
for i, (key, values) in enumerate(ensemble_members_inv.items()):
    rmse = np.sqrt(mean_squared_error(actual_temp_actual, values))
    mae = mean_absolute_error(actual_temp_actual, values)
    print(f"Member {i+1}: RMSE = {rmse:.4f}°C, MAE = {mae:.4f}°C")
print(f"\nEnsemble Average: RMSE = {overall_rmse:.4f}°C, MAE = {overall_mae:.4f}°C")
print("="*60)


In [ ]:
# ============================================================================
# Final Summary and Save Results
# ============================================================================

print("="*80)
print("DUNE Temperature Forecasting Model - Final Summary")
print("="*80)
print(f"\nLocation: Rangpur, Bangladesh (lat={target_lat}, lon={target_lon})")
print(f"\nData Configuration:")
print(f"  - Full data range: 1995-2024")
print(f"  - Target months: March-July")
print(f"  - Training period: 1995-2019")
print(f"  - Validation period: 2020-2022")
print(f"  - Test period: 2024 (March-July)")
print(f"\nModel Architecture: DUNE1D (Deep UNet++ Ensemble for 1D Time Series)")
print(f"  - Encoder levels: 3")
print(f"  - Decoder levels: 3")
print(f"  - Bottleneck filters: {32*8}")
print(f"  - Ensemble members: 4")
print(f"  - Sequence length: {SEQ_LENGTH}")
print(f"  - Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining Configuration:")
print(f"  - Epochs trained: {len(history['train_loss'])}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Initial learning rate: {LEARNING_RATE}")
print(f"  - Optimizer: Adam")
print(f"  - Scheduler: CosineAnnealingLR")
print(f"  - Loss: Combined (RMSE + MAE + Ensemble Consistency)")
print(f"\n{'='*80}")
print("2024 Forecast Performance (March-July)")
print("="*80)
print(f"\nAbsolute Temperature Metrics:")
print(f"  - RMSE:  {overall_rmse:.4f}°C")
print(f"  - MAE:   {overall_mae:.4f}°C")
print(f"  - R²:    {overall_r2:.4f}")
print(f"  - MAPE:  {overall_mape:.2f}%")
print(f"  - ACC:   {overall_acc:.4f}")
print(f"\nTemperature Anomaly Metrics:")
print(f"  - RMSE:  {anomaly_rmse:.4f}°C")
print(f"  - MAE:   {anomaly_mae:.4f}°C")
print(f"  - ACC:   {anomaly_acc:.4f}")
print(f"\nTemperature Statistics:")
print(f"  - Actual Mean: {np.mean(actual_temp_actual):.2f}°C")
print(f"  - Predicted Mean: {np.mean(pred_temp_actual):.2f}°C")
print(f"  - Actual Range: [{actual_temp_actual.min():.2f}, {actual_temp_actual.max():.2f}]°C")
print(f"  - Predicted Range: [{pred_temp_actual.min():.2f}, {pred_temp_actual.max():.2f}]°C")
print("="*80)

# Save predictions to CSV
results_df.to_csv(os.path.join(OUTPUT_DIR, 'dune_predictions_2024.csv'), index=False)
print(f"\nPredictions saved to 'dune_predictions_2024.csv'")

# Save monthly metrics
monthly_df.to_csv(os.path.join(OUTPUT_DIR, 'dune_monthly_metrics.csv'), index=False)
print(f"Monthly metrics saved to 'dune_monthly_metrics.csv'")

# Save final model
final_model_path = os.path.join(OUTPUT_DIR, 'dune_model_rangpur.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'scaler_X': scaler_X,
    'scaler_y': scaler_y,
    'climatology': climatology,
    'history': history,
    'config': {
        'in_channels': INPUT_CHANNELS,
        'seq_length': SEQ_LENGTH,
        'base_filters': 32,
        'features': FEATURES,
        'target': TARGET
    },
    'metrics': {
        'rmse': overall_rmse,
        'mae': overall_mae,
        'r2': overall_r2,
        'acc': overall_acc
    }
}, final_model_path)
print(f"Model saved to '{final_model_path}'")

# List all saved files
print(f"\n{'='*80}")
print("All Saved Files:")
print("="*80)
saved_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith('dune_')]
for f in sorted(saved_files):
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {f}: {size_kb:.2f} KB")


In [ ]:
# ============================================================================
# Display Predictions Table by Month
# ============================================================================

print("\n" + "="*100)
print("Detailed 2024 Predictions by Month")
print("="*100)

for month in TARGET_MONTHS:
    month_data = results_df[results_df['Month'] == month].copy()
    if len(month_data) > 0:
        print(f"\n{MONTH_NAMES[month]} 2024:")
        print("-"*80)
        display_df = month_data[['Date', 'Actual_Temp', 'Predicted_Temp', 'Error', 'Abs_Error']].copy()
        display_df['Date'] = display_df['Date'].dt.strftime('%Y-%m-%d')
        display_df = display_df.round(3)
        print(display_df.to_string(index=False))
        
        # Month summary
        m_rmse = np.sqrt(mean_squared_error(month_data['Actual_Temp'], month_data['Predicted_Temp']))
        m_mae = mean_absolute_error(month_data['Actual_Temp'], month_data['Predicted_Temp'])
        print(f"\n{MONTH_NAMES[month]} Summary: RMSE={m_rmse:.3f}°C, MAE={m_mae:.3f}°C, "
              f"Mean Error={month_data['Error'].mean():.3f}°C")
        print("="*80)


# DUNE Model Implementation - Rangpur Temperature Forecasting

## Overview
This notebook implements the **DUNE (Deep UNet++ based Ensemble)** model for temperature forecasting in Rangpur, Bangladesh. The model is adapted from the original DUNE architecture designed for spatial climate data to work with 1D time series data.

## Key Features
- **Residual Blocks**: CNN blocks with skip connections for better gradient flow
- **UNet++ Architecture**: Encoder-decoder structure with multi-scale feature extraction
- **Ensemble Approach**: 4-member ensemble for uncertainty quantification
- **Anomaly-based Forecasting**: Predicts temperature anomalies from climatology

## Data Configuration
- **Source**: ERA5 reanalysis data (`instant_all.nc`)
- **Variables**: t2m (2m temperature), swvl1 (soil moisture)
- **Time Range**: 1995-2024
- **Target Months**: March-July
- **Training**: 1995-2019
- **Validation**: 2020-2022
- **Testing**: 2024 (March-July)

## Reference
Based on: *DUNE: A Machine Learning Deep UNet++ Based Ensemble Approach to Monthly, Seasonal and Annual Climate Forecasting* by Shukla & Halem (2024)
